# Modelo MLP de Predicción de Deserción — V.2 Experimental

**Objetivo:** implementar y evaluar una versión V.2 del MLP siguiendo las observaciones del tutor, manteniendo los mismos conjuntos preparados de **TRAIN, VALIDATION y TEST** utilizados por V.1.

## Principios metodológicos de esta versión

- V.1 se conserva intacta como **baseline**.
- V.2 utiliza exactamente los mismos archivos `_ready.csv` para que la comparación sea justa.
- El preprocesamiento fue validado por separado: el pipeline se ajusta únicamente con TRAIN y luego aplica `transform()` a VALIDATION y TEST.
- La selección de thresholds se realiza **exclusivamente con VALIDATION**.
- TEST se utiliza únicamente después de congelar los thresholds.
- Los resultados de una sola ejecución siguen considerándose **experimentales/preliminares**.
- La V.2 no se declarará superior hasta compararla formalmente con V.1 y realizar análisis posteriores de estabilidad.

## Cambios V.1 → V.2

1. `binary_crossentropy + class_weight` → **Focal Loss** (`gamma=2.0`, `alpha` calculado desde TRAIN).
2. Dropout `0.35 / 0.30 / 0.20` → **`0.25 / 0.20 / 0.10`**.
3. L2 se mantiene únicamente en las capas de **256 y 128** neuronas.
4. `batch_size=32` → **64**.
5. EarlyStopping `patience=20` → **12**.
6. ReduceLROnPlateau `patience=8` → **6**.
7. Threshold fijo 0.50 → comparación de:
   - Baseline 0.50
   - Threshold óptimo por F1 en VALIDATION
   - Threshold con Recall ≥ 0.80 en VALIDATION, maximizando Precision entre los candidatos válidos
8. Se incorporan:
   - MCC
   - Balanced Accuracy
   - FNR
   - Curva Precision-Recall
   - Curva de calibración
   - Brier Score


## 1) Librerías, reproducibilidad y Google Drive


In [ ]:
import os
import json
import math
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    ConfusionMatrixDisplay
)

from sklearn.calibration import calibration_curve

SEED = 42

# Semillas para Python/NumPy/TensorFlow.
tf.keras.utils.set_random_seed(SEED)

# Intentar operaciones deterministas cuando el entorno lo permita.
try:
    tf.config.experimental.enable_op_determinism()
    print("Determinismo de TensorFlow activado.")
except Exception as e:
    print("No fue posible activar determinismo completo:", e)

# Versión pública: se omite el montaje automático de Drive personal.


## 2) Carga de los mismos datos preparados utilizados por V.1


In [ ]:
DATOS_PROCESADOS = Path("data/procesados")
train_path = DATOS_PROCESADOS / "REGEN_basev6_X_train_ready.csv"
val_path   = DATOS_PROCESADOS / "REGEN_basev6_X_val_ready.csv"
test_path  = DATOS_PROCESADOS / "REGEN_basev6_X_test_ready.csv"

train_df = pd.read_csv(train_path, encoding="utf-8-sig")
val_df   = pd.read_csv(val_path, encoding="utf-8-sig")
test_df  = pd.read_csv(test_path, encoding="utf-8-sig")

print("TRAIN:", train_df.shape)
print("VAL  :", val_df.shape)
print("TEST :", test_df.shape)

## 3) Validaciones de integridad antes del entrenamiento

Estas validaciones buscan detectar problemas de columnas, orden, valores faltantes, infinitos o clases inesperadas antes de entrenar V.2.


In [ ]:
# Validar existencia del target.
for nombre, df in [("TRAIN", train_df), ("VAL", val_df), ("TEST", test_df)]:
    if "y" not in df.columns:
        raise ValueError(f"{nombre} no contiene la columna objetivo 'y'.")

# Columnas predictoras.
feature_cols = [c for c in train_df.columns if c != "y"]

if len(feature_cols) != len(set(feature_cols)):
    raise ValueError("TRAIN contiene nombres de variables duplicados.")

# Deben tener exactamente el mismo conjunto de features.
for nombre, df in [("VAL", val_df), ("TEST", test_df)]:
    cols = [c for c in df.columns if c != "y"]

    faltantes = sorted(set(feature_cols) - set(cols))
    extras = sorted(set(cols) - set(feature_cols))

    if faltantes or extras:
        raise ValueError(
            f"{nombre} no coincide con TRAIN. "
            f"Faltantes={faltantes[:10]}, Extras={extras[:10]}"
        )

# Reordenar explícitamente VAL/TEST según TRAIN para garantizar orden idéntico.
train_df = train_df[feature_cols + ["y"]]
val_df   = val_df[feature_cols + ["y"]]
test_df  = test_df[feature_cols + ["y"]]

# Separar X / y.
y_train = train_df["y"].astype(int).to_numpy()
X_train = train_df[feature_cols].astype(np.float32).to_numpy()

y_val = val_df["y"].astype(int).to_numpy()
X_val = val_df[feature_cols].astype(np.float32).to_numpy()

y_test = test_df["y"].astype(int).to_numpy()
X_test = test_df[feature_cols].astype(np.float32).to_numpy()

# Clases binarias.
for nombre, y in [("TRAIN", y_train), ("VAL", y_val), ("TEST", y_test)]:
    clases = set(np.unique(y))
    if not clases.issubset({0, 1}) or len(clases) != 2:
        raise ValueError(f"{nombre}: clases inesperadas {clases}. Se esperaban 0 y 1.")

# NaN / infinitos.
for nombre, X in [("TRAIN", X_train), ("VAL", X_val), ("TEST", X_test)]:
    if np.isnan(X).any():
        raise ValueError(f"{nombre} contiene valores NaN después del preprocesamiento.")
    if not np.isfinite(X).all():
        raise ValueError(f"{nombre} contiene valores infinitos.")

# Dimensiones compatibles.
if not (X_train.shape[1] == X_val.shape[1] == X_test.shape[1]):
    raise ValueError("TRAIN/VAL/TEST tienen distinto número de features.")

print("Validaciones de datos completadas correctamente.")
print("Número de features:", X_train.shape[1])
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

for nombre, y in [("TRAIN", y_train), ("VAL", y_val), ("TEST", y_test)]:
    n = len(y)
    n0 = int(np.sum(y == 0))
    n1 = int(np.sum(y == 1))
    print(
        f"{nombre}: Permanencia={n0} ({n0/n:.2%}) | "
        f"Deserción={n1} ({n1/n:.2%})"
    )


## 4) Focal Loss V.2

`alpha` se calcula **solo a partir de TRAIN**.  
Con `y=1` para Deserción, `alpha = 1 - mean(y_train)` asigna mayor peso relativo a la clase positiva cuando esta es minoritaria.

Se implementa como una clase serializable de Keras para permitir guardar y volver a cargar correctamente el modelo `.keras`.


In [ ]:
@tf.keras.utils.register_keras_serializable(package="TFG")
class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name="binary_focal_loss"):
        super().__init__(name=name)
        self.gamma = float(gamma)
        self.alpha = float(alpha)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)

        # Evita broadcasting accidental entre (batch,) y (batch, 1).
        y_true = tf.reshape(y_true, tf.shape(y_pred))

        eps = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)

        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)

        p_t = (
            y_true * y_pred
            + (1.0 - y_true) * (1.0 - y_pred)
        )

        alpha_factor = (
            y_true * self.alpha
            + (1.0 - y_true) * (1.0 - self.alpha)
        )

        modulating_factor = tf.pow(1.0 - p_t, self.gamma)

        loss = alpha_factor * modulating_factor * bce

        # Retornar pérdida por observación; Keras aplica la reducción final.
        return tf.reduce_mean(loss, axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({
            "gamma": self.gamma,
            "alpha": self.alpha
        })
        return config


positive_rate_train = float(np.mean(y_train))
alpha_v2 = float(1.0 - positive_rate_train)
gamma_v2 = 2.0

print(f"Proporción Deserción en TRAIN: {positive_rate_train:.4f}")
print(f"alpha Focal Loss: {alpha_v2:.4f}")
print(f"gamma Focal Loss: {gamma_v2:.1f}")

if not (0.0 < alpha_v2 < 1.0):
    raise ValueError("alpha calculado está fuera del intervalo (0,1).")


## 5) Arquitectura MLP V.2


In [ ]:
# Reiniciar semillas justo antes de construir el modelo.
tf.keras.utils.set_random_seed(SEED)

model_v2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),

    tf.keras.layers.Dense(
        256,
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Dense(
        128,
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(64),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.10),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=BinaryFocalLoss(
        gamma=gamma_v2,
        alpha=alpha_v2
    ),
    metrics=[
        tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)

model_v2.summary()


## 6) Callbacks V.2


In [ ]:
callbacks_v2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_pr_auc",
        mode="max",
        factor=0.5,
        patience=6,
        min_lr=1e-6,
        verbose=1
    )
]


## 7) Entrenamiento V.2

**Importante:** V.2 ya no utiliza `class_weight`; el tratamiento del desbalance se realiza mediante Focal Loss.


In [ ]:
history_v2 = model_v2.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=64,
    callbacks=callbacks_v2,
    verbose=1
)


## 8) Diagnóstico del entrenamiento


In [ ]:
required_history = ["loss", "val_loss", "pr_auc", "val_pr_auc"]

for key in required_history:
    if key not in history_v2.history:
        raise KeyError(f"No se encontró '{key}' en history_v2.history.")

train_loss = np.asarray(history_v2.history["loss"], dtype=float)
val_loss = np.asarray(history_v2.history["val_loss"], dtype=float)
train_pr_auc = np.asarray(history_v2.history["pr_auc"], dtype=float)
val_pr_auc = np.asarray(history_v2.history["val_pr_auc"], dtype=float)

best_val_loss_epoch = int(np.argmin(val_loss)) + 1
best_val_loss = float(np.min(val_loss))

best_val_pr_epoch = int(np.argmax(val_pr_auc)) + 1
best_val_pr_auc = float(np.max(val_pr_auc))

diagnostico_entrenamiento = pd.DataFrame({
    "Indicador": [
        "Épocas ejecutadas",
        "Época con menor val_loss",
        "val_loss mínima",
        "Época con mayor val_PR-AUC",
        "val_PR-AUC máxima",
        "Coincidencia de épocas"
    ],
    "Valor": [
        len(train_loss),
        best_val_loss_epoch,
        best_val_loss,
        best_val_pr_epoch,
        best_val_pr_auc,
        "Sí" if best_val_loss_epoch == best_val_pr_epoch else "No"
    ]
})

display(diagnostico_entrenamiento)

print(
    "Nota: no es obligatorio que la menor val_loss y el mayor "
    "val_PR-AUC ocurran en la misma época."
)
print(
    "EarlyStopping restaura los pesos asociados al mayor val_PR-AUC, "
    "porque ese es el criterio configurado."
)


In [ ]:
# Curva de pérdida
epochs_hist = np.arange(1, len(train_loss) + 1)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(epochs_hist, train_loss, label="Entrenamiento", linewidth=2.2)
ax.plot(epochs_hist, val_loss, label="Validación", linewidth=2.2)
ax.axvline(
    best_val_loss_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Menor val_loss (época {best_val_loss_epoch})"
)
ax.scatter([best_val_loss_epoch], [best_val_loss], s=55, zorder=3)
ax.set_xlabel("Época")
ax.set_ylabel("Focal Loss + regularización")
ax.set_title("V.2 — Curva de pérdida")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# Curva PR-AUC de entrenamiento
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(epochs_hist, train_pr_auc, label="Entrenamiento", linewidth=2.2)
ax.plot(epochs_hist, val_pr_auc, label="Validación", linewidth=2.2)
ax.axvline(
    best_val_pr_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Mayor val_PR-AUC (época {best_val_pr_epoch})"
)
ax.scatter([best_val_pr_epoch], [best_val_pr_auc], s=55, zorder=3)
ax.set_xlabel("Época")
ax.set_ylabel("PR-AUC")
ax.set_title("V.2 — Evolución del PR-AUC")
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


# 9) Selección de thresholds — **SOLO VALIDATION**

En esta sección aún **no se consulta TEST**.

Se seleccionan:
1. `0.50` como baseline.
2. Threshold que maximiza F1 en VALIDATION.
3. Threshold que cumple `Recall >= 0.80` y, entre esos candidatos, maximiza Precision.

La tercera regla evita escoger innecesariamente el primer threshold muy bajo que cumpla Recall.


In [ ]:
# Predicciones únicamente sobre VALIDATION.
y_prob_val = model_v2.predict(X_val, verbose=0).ravel()

if not np.all(np.isfinite(y_prob_val)):
    raise ValueError("VALIDATION produjo probabilidades no finitas.")

if not np.all((y_prob_val >= 0.0) & (y_prob_val <= 1.0)):
    raise ValueError("VALIDATION produjo probabilidades fuera de [0,1].")

prec_val, rec_val, thresholds_val = precision_recall_curve(
    y_val,
    y_prob_val
)

if thresholds_val.size == 0:
    raise RuntimeError(
        "No fue posible construir thresholds desde VALIDATION."
    )

# precision_recall_curve devuelve precision/recall con un elemento extra.
p_thr = prec_val[:-1]
r_thr = rec_val[:-1]

f1_thr = (
    2.0 * p_thr * r_thr
    / (p_thr + r_thr + 1e-12)
)

best_f1_idx = int(np.nanargmax(f1_thr))
best_thresh_f1 = float(thresholds_val[best_f1_idx])

def threshold_for_target_recall(
    thresholds,
    precisions,
    recalls,
    target_recall
):
    valid_idx = np.where(recalls >= target_recall)[0]

    if len(valid_idx) == 0:
        return {
            "threshold": np.nan,
            "precision_val": np.nan,
            "recall_val": np.nan,
            "index": None
        }

    candidate_precisions = precisions[valid_idx]
    max_precision = np.max(candidate_precisions)

    # Resolver empates eligiendo el threshold más alto entre
    # los candidatos con la mejor precision que todavía cumplen Recall.
    tie_idx = valid_idx[
        np.isclose(candidate_precisions, max_precision)
    ]

    chosen_idx = int(
        tie_idx[np.argmax(thresholds[tie_idx])]
    )

    return {
        "threshold": float(thresholds[chosen_idx]),
        "precision_val": float(precisions[chosen_idx]),
        "recall_val": float(recalls[chosen_idx]),
        "index": chosen_idx
    }


TARGET_RECALL = 0.80

recall80_selection = threshold_for_target_recall(
    thresholds_val,
    p_thr,
    r_thr,
    TARGET_RECALL
)

if math.isnan(recall80_selection["threshold"]):
    raise RuntimeError(
        f"No existe un threshold en VALIDATION que alcance Recall >= {TARGET_RECALL:.2f}."
    )

best_thresh_recall80 = recall80_selection["threshold"]

thresholds_seleccionados = {
    "Baseline 0.50": 0.50,
    "F1-opt VAL": best_thresh_f1,
    "Recall80-opt VAL": best_thresh_recall80
}

print("THRESHOLDS CONGELADOS DESDE VALIDATION")
for name, threshold in thresholds_seleccionados.items():
    print(f"{name}: {threshold:.6f}")

print(
    f"\nF1-opt VAL -> "
    f"Precision={p_thr[best_f1_idx]:.4f}, "
    f"Recall={r_thr[best_f1_idx]:.4f}, "
    f"F1={f1_thr[best_f1_idx]:.4f}"
)

print(
    f"Recall80-opt VAL -> "
    f"Precision={recall80_selection['precision_val']:.4f}, "
    f"Recall={recall80_selection['recall_val']:.4f}"
)


## 10) Tabla de política de Recall en VALIDATION


In [ ]:
# Diagnóstico adicional: no modifica los tres escenarios principales.
policy_rows = []

for target in [0.80, 0.85, 0.90, 0.95]:
    selection = threshold_for_target_recall(
        thresholds_val,
        p_thr,
        r_thr,
        target
    )

    policy_rows.append({
        "Recall objetivo": target,
        "Threshold VAL": selection["threshold"],
        "Precision VAL": selection["precision_val"],
        "Recall obtenido VAL": selection["recall_val"]
    })

recall_policy_val = pd.DataFrame(policy_rows)
display(recall_policy_val.style.format({
    "Recall objetivo": "{:.2f}",
    "Threshold VAL": "{:.4f}",
    "Precision VAL": "{:.4f}",
    "Recall obtenido VAL": "{:.4f}"
}))


In [ ]:
# Comportamiento de Precision / Recall / F1 según threshold en VALIDATION
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(thresholds_val, p_thr, label="Precision", linewidth=2)
ax.plot(thresholds_val, r_thr, label="Recall", linewidth=2)
ax.plot(thresholds_val, f1_thr, label="F1", linewidth=2)

for name, threshold in thresholds_seleccionados.items():
    ax.axvline(
        threshold,
        linestyle="--",
        alpha=0.45,
        label=f"{name}: {threshold:.3f}"
    )

ax.set_xlabel("Threshold")
ax.set_ylabel("Métrica en VALIDATION")
ax.set_title("V.2 — Selección de threshold en VALIDATION")
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


# 11) Evaluación final en TEST

A partir de este punto los thresholds ya quedaron congelados.  
TEST no participa en la selección del umbral.


In [ ]:
# TEST se consulta por primera vez después de congelar thresholds.
test_eval_keras = model_v2.evaluate(
    X_test,
    y_test,
    verbose=0,
    return_dict=True
)

print("Métricas Keras TEST:")
print(test_eval_keras)

y_prob_test = model_v2.predict(
    X_test,
    verbose=0
).ravel()

if not np.all(np.isfinite(y_prob_test)):
    raise ValueError("TEST produjo probabilidades no finitas.")

if not np.all((y_prob_test >= 0.0) & (y_prob_test <= 1.0)):
    raise ValueError("TEST produjo probabilidades fuera de [0,1].")


## 12) Métricas globales independientes del threshold


In [ ]:
roc_auc_test = float(
    roc_auc_score(y_test, y_prob_test)
)

ap_test = float(
    average_precision_score(y_test, y_prob_test)
)

precision_curve_test, recall_curve_test, _ = precision_recall_curve(
    y_test,
    y_prob_test
)

pr_auc_test = float(
    auc(recall_curve_test, precision_curve_test)
)

brier_test = float(
    brier_score_loss(y_test, y_prob_test)
)

metricas_probabilisticas = pd.DataFrame({
    "Métrica": [
        "ROC-AUC TEST",
        "PR-AUC TEST (área trapezoidal)",
        "Average Precision (AP) TEST",
        "Brier Score TEST"
    ],
    "Valor": [
        roc_auc_test,
        pr_auc_test,
        ap_test,
        brier_test
    ]
})

display(metricas_probabilisticas.style.format({"Valor": "{:.4f}"}))


## 13) Evaluación de los tres escenarios de threshold


In [ ]:
def evaluate_threshold_scenario(
    name,
    threshold,
    y_true,
    y_prob
):
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    return {
        "Escenario": name,
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_Desercion": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall_Desercion": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1_Desercion": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "Specificity": specificity,
        "FNR": fnr,
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(
            y_true, y_pred
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Correctos": int(tn + tp),
        "Errores": int(fp + fn),
        "y_pred": y_pred,
        "cm": cm
    }


scenario_results = []

for name, threshold in thresholds_seleccionados.items():
    scenario_results.append(
        evaluate_threshold_scenario(
            name,
            threshold,
            y_test,
            y_prob_test
        )
    )

scenario_table = pd.DataFrame([
    {
        k: v
        for k, v in result.items()
        if k not in {"y_pred", "cm"}
    }
    for result in scenario_results
])

display(
    scenario_table.style.format({
        "Threshold": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Precision_Desercion": "{:.4f}",
        "Recall_Desercion": "{:.4f}",
        "F1_Desercion": "{:.4f}",
        "Specificity": "{:.4f}",
        "FNR": "{:.4f}",
        "Balanced_Accuracy": "{:.4f}",
        "MCC": "{:.4f}"
    })
)

print(
    "\nIMPORTANTE: no seleccionar el 'mejor' escenario mirando TEST. "
    "Los thresholds ya fueron definidos con VALIDATION; TEST solo reporta "
    "cómo se comporta cada política."
)


## 14) Reporte y matriz de confusión por escenario


In [ ]:
for result in scenario_results:
    name = result["Escenario"]
    threshold = result["Threshold"]
    y_pred = result["y_pred"]

    print("\n" + "=" * 78)
    print(f"{name} | threshold={threshold:.6f}")
    print("=" * 78)

    print("Matriz de confusión:")
    print(result["cm"])

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Permanencia", "Deserción"],
            digits=4,
            zero_division=0
        )
    )

    print(
        f"MCC={result['MCC']:.4f} | "
        f"Balanced Accuracy={result['Balanced_Accuracy']:.4f} | "
        f"FNR={result['FNR']:.4f}"
    )


In [ ]:
# Una figura independiente para cada escenario.
for result in scenario_results:
    fig, ax = plt.subplots(figsize=(6.8, 5.8))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=result["cm"],
        display_labels=["Permanencia", "Deserción"]
    )

    disp.plot(
        ax=ax,
        values_format="d",
        colorbar=False
    )

    ax.set_title(
        f"V.2 — {result['Escenario']}\n"
        f"Threshold={result['Threshold']:.3f}"
    )
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Condición real")

    plt.tight_layout()
    plt.show()


## 15) Curvas ROC y Precision-Recall en TEST


In [ ]:
# Curva ROC
fpr_test, tpr_test, _ = roc_curve(
    y_test,
    y_prob_test
)

fig, ax = plt.subplots(figsize=(7.5, 6))
ax.plot(
    fpr_test,
    tpr_test,
    linewidth=2.3,
    label=f"MLP V.2 (ROC-AUC={roc_auc_test:.4f})"
)
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Aleatorio"
)
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("V.2 — Curva ROC en TEST")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# Curva Precision-Recall
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.plot(
    recall_curve_test,
    precision_curve_test,
    linewidth=2.3,
    label=(
        f"MLP V.2 "
        f"(PR-AUC={pr_auc_test:.4f}, AP={ap_test:.4f})"
    )
)

baseline_positive_rate = float(np.mean(y_test))
ax.axhline(
    baseline_positive_rate,
    linestyle="--",
    label=f"Prevalencia TEST={baseline_positive_rate:.3f}"
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("V.2 — Curva Precision-Recall en TEST")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 16) Calibración de probabilidades

Esta sección **evalúa** calibración sobre TEST, pero no reajusta el modelo ni recalibra probabilidades usando TEST.

Se utiliza `strategy="quantile"` para obtener bins con cantidades de observaciones más comparables, lo cual resulta útil con un TEST relativamente pequeño.


In [ ]:
prob_true, prob_pred = calibration_curve(
    y_test,
    y_prob_test,
    n_bins=10,
    strategy="quantile"
)

fig, ax = plt.subplots(figsize=(7.2, 6))
ax.plot(
    prob_pred,
    prob_true,
    marker="o",
    linewidth=2,
    label="MLP V.2"
)
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Calibración ideal"
)
ax.set_xlabel("Probabilidad predicha de deserción")
ax.set_ylabel("Proporción observada de deserción")
ax.set_title(
    f"V.2 — Curva de calibración\n"
    f"Brier Score={brier_test:.4f}"
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Brier Score TEST: {brier_test:.6f}")
print(
    "Interpretación: menor Brier Score implica menor error cuadrático "
    "de las probabilidades; debe interpretarse junto con la curva."
)


## 17) Comparación opcional V.1 vs V.2

Para aislar el efecto de arquitectura/loss, la comparación directa con V.1 debe hacerse preferiblemente usando **el mismo threshold 0.50**.  
La optimización de threshold de V.2 se reporta por separado como decisión operativa.

Si el snapshot de V.1 está disponible en Colab, esta celda construye automáticamente la comparación.


In [ ]:
# Ajusta esta ruta si guardaste el snapshot de V.1 en Drive.
V1_SNAPSHOT_PATH = str(Path("outputs/resultados_mlp_v1") / "snapshot_resultados_preliminares.json")

v2_baseline = next(
    r for r in scenario_results
    if r["Escenario"] == "Baseline 0.50"
)

if os.path.exists(V1_SNAPSHOT_PATH):
    with open(
        V1_SNAPSHOT_PATH,
        "r",
        encoding="utf-8"
    ) as f:
        v1_snapshot = json.load(f)

    v1_metrics = v1_snapshot.get("metricas", {})

    comparacion_v1_v2 = pd.DataFrame({
        "Métrica": [
            "Accuracy",
            "Precision deserción",
            "Recall deserción",
            "F1 deserción",
            "Specificity",
            "FNR",
            "Balanced Accuracy",
            "MCC",
            "ROC-AUC"
        ],
        "V.1 threshold 0.50": [
            v1_metrics.get("accuracy", np.nan),
            v1_metrics.get("precision_desercion", np.nan),
            v1_metrics.get("recall_desercion", np.nan),
            v1_metrics.get("f1_desercion", np.nan),
            v1_metrics.get("specificity", np.nan),
            v1_metrics.get("false_negative_rate", np.nan),
            v1_metrics.get("balanced_accuracy", np.nan),
            v1_metrics.get("mcc", np.nan),
            v1_metrics.get("roc_auc", np.nan)
        ],
        "V.2 threshold 0.50": [
            v2_baseline["Accuracy"],
            v2_baseline["Precision_Desercion"],
            v2_baseline["Recall_Desercion"],
            v2_baseline["F1_Desercion"],
            v2_baseline["Specificity"],
            v2_baseline["FNR"],
            v2_baseline["Balanced_Accuracy"],
            v2_baseline["MCC"],
            roc_auc_test
        ]
    })

    display(
        comparacion_v1_v2.style.format({
            "V.1 threshold 0.50": "{:.4f}",
            "V.2 threshold 0.50": "{:.4f}"
        })
    )
else:
    print(
        "Snapshot V.1 no encontrado. "
        "La comparación automática se omitió."
    )
    print(
        "Esto NO afecta el entrenamiento ni la evaluación de V.2."
    )


## 18) Exportación completa de resultados V.2


In [ ]:
OUTPUT_ROOT = Path("outputs/resultados_mlp_v2")

PLOTS_DIR = OUTPUT_ROOT / "graficos"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Tablas.
scenario_table.drop(
    columns=[],
    errors="ignore"
).to_csv(
    OUTPUT_ROOT / "metricas_thresholds_test.csv",
    index=False,
    encoding="utf-8-sig"
)

metricas_probabilisticas.to_csv(
    OUTPUT_ROOT / "metricas_probabilisticas_test.csv",
    index=False,
    encoding="utf-8-sig"
)

diagnostico_entrenamiento.to_csv(
    OUTPUT_ROOT / "diagnostico_entrenamiento_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

recall_policy_val.to_csv(
    OUTPUT_ROOT / "politica_recall_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

# Snapshot estructurado.
snapshot_v2 = {
    "version": "MLP_V2_EXPERIMENTAL",
    "fecha_ejecucion": datetime.now().isoformat(),
    "seed": SEED,
    "data_paths": {
        "train": train_path,
        "validation": val_path,
        "test": test_path
    },
    "n_features": int(X_train.shape[1]),
    "class_distribution": {
        "train_positive_rate": float(np.mean(y_train)),
        "validation_positive_rate": float(np.mean(y_val)),
        "test_positive_rate": float(np.mean(y_test))
    },
    "focal_loss": {
        "gamma": gamma_v2,
        "alpha": alpha_v2
    },
    "training": {
        "batch_size": 64,
        "early_stopping_monitor": "val_pr_auc",
        "early_stopping_patience": 12,
        "reduce_lr_patience": 6,
        "epochs_executed": int(len(train_loss)),
        "best_val_loss_epoch": best_val_loss_epoch,
        "best_val_loss": best_val_loss,
        "best_val_pr_auc_epoch": best_val_pr_epoch,
        "best_val_pr_auc": best_val_pr_auc
    },
    "thresholds_selected_on_validation": {
        k: float(v)
        for k, v in thresholds_seleccionados.items()
    },
    "test_probability_metrics": {
        "roc_auc": roc_auc_test,
        "pr_auc_trapezoidal": pr_auc_test,
        "average_precision": ap_test,
        "brier_score": brier_test
    },
    "test_threshold_scenarios": [
        {
            k: (
                v.item()
                if isinstance(v, np.generic)
                else v
            )
            for k, v in result.items()
            if k not in {"y_pred", "cm"}
        }
        for result in scenario_results
    ],
    "methodological_note": (
        "Thresholds fueron seleccionados exclusivamente con VALIDATION. "
        "TEST se utilizó posteriormente solo para evaluación. "
        "Resultados correspondientes a una ejecución experimental."
    )
}

with open(
    OUTPUT_ROOT / "snapshot_mlp_v2_experimental.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        snapshot_v2,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Resultados exportados en:", OUTPUT_ROOT)


## 19) Guardar modelo V.2


In [ ]:
MODEL_PATH = str(OUTPUT_ROOT / "modelo_desercion_mlp_v2.keras")

model_v2.save(MODEL_PATH)

print("Modelo V.2 guardado en:")
print(MODEL_PATH)
print(
    "La clase BinaryFocalLoss está registrada como serializable "
    "para facilitar la carga posterior del modelo."
)


## 20) Próximos pasos después de validar esta V.2

1. Comparar formalmente V.1 vs V.2 usando el mismo TEST y threshold 0.50 para aislar cambios del modelo.
2. Analizar por separado la utilidad operativa de los thresholds seleccionados con VALIDATION.
3. Ejecutar 5-fold estratificado y reportar media ± desviación de PR-AUC, Recall, MCC y otras métricas.
4. Evaluar explicabilidad con SHAP únicamente cuando se haya definido qué configuración continúa a la etapa final.
5. No recalibrar ni ajustar nuevos hiperparámetros utilizando TEST.
